In [2]:
# Đây là mẫu ví dụ template:
data_exmp = [
    {
        "template": "PacketResponder <*> for block <*> terminating",
        "message": "PacketResponder 1 for block blk_1073741825 terminating"
    },
    {
        "template": "BLOCK* NameSystem.addStoredBlock: blockMap updated: <*>:<*> is added to <*> size <*>",
        "message": "BLOCK* NameSystem.addStoredBlock: blockMap updated: 127.0.0.1:50010 is added to blockpool size 67108864"
    },
    {
        "template": "Received block <*> of size <*> from <*>",
        "message": "Received block blk_1073741827 of size 1048576 from 127.0.0.1"
    },
    {
        "template": "Receiving block <*> src: <*>:<*> dest: <*>:<*>",
        "message": "Receiving block blk_1073741827 src: 127.0.0.1:50010 dest: 127.0.0.1:50020"
    },
    {
        "template": "BLOCK* NameSystem.allocateBlock: <*> <*>",
        "message": "BLOCK* NameSystem.allocateBlock: user1 blk_1073741826"
    },
    {
        "template": "Verification succeeded for <*>",
        "message": "Verification succeeded for blk_1073741828"
    },
    {
        "template": "Deleting block <*> file <*>",
        "message": "Deleting block blk_1073741829 file /tmp/hadoop/blk_1073741829"
    },
    {
        "template": "<*>:<*> Served block <*> to <*>",
        "message": "127.0.0.1:50010 Served block blk_1073741830 to client1"
    },
    {
        "template": "<*>:<*>:Got exception while serving <*> to <*>:",
        "message": "127.0.0.1:50010:Got exception while serving blk_1073741831 to client2:"
    },
    {
        "template": "BLOCK* NameSystem.delete: <*> is added to invalidSet of <*>:<*>",
        "message": "BLOCK* NameSystem.delete: blk_1073741832 is added to invalidSet of 127.0.0.1:50010"
    },
    {
        "template": "<*>:<*> Starting thread to transfer block <*> to <*>:<*>",
        "message": "127.0.0.1:50010 Starting thread to transfer block blk_1073741833 to 127.0.0.1:50011"
    },
    {
        "template": "BLOCK* ask <*>:<*> to delete <*>",
        "message": "BLOCK* ask 127.0.0.1:50010 to delete blk_1073741834"
    },
    {
        "template": "Received block <*> src: <*>:<*> dest: <*>:<*> of size <*>",
        "message": "Received block blk_1073741835 src: 127.0.0.1:50010 dest: 127.0.0.1:50020 of size 1048576"
    },
    {
        "template": "BLOCK* ask <*>:<*> to replicate <*> to datanode(s) <*>:<*>",
        "message": "BLOCK* ask 127.0.0.1:50010 to replicate blk_1073741836 to datanode(s) 127.0.0.1:50012"
    }
]


In [ ]:
# File GA_calculator.py trong UNLEASH
import pandas as pd
from tqdm import tqdm

def evaluate(df_groundtruth, df_parsedlog, filter_templates=None):
    """ Đánh giá độ chính xác của việc phân tích log bằng cách so sánh dữ liệu ground truth và kết quả phân tích log. Phương thức này thực hiện các bước sau:
        1. Loại bỏ các dòng không hợp lệ trong dữ liệu ground truth (các dòng có giá trị NaN trong cột `EventTemplate`).
        2. Gọi hàm `get_accuracy` để tính toán các chỉ số độ chính xác:
            - GA (Grouping Accuracy): Độ chính xác nhóm.
            - FGA (F-Measure of Grouping Accuracy): Trung bình điều hòa giữa độ chính xác và độ bao phủ.
        3. In kết quả GA và FGA ra màn hình.
        4. Trả về giá trị GA và FGA.

    Args:
        df_groundtruth (pd.DataFrame): DataFrame chứa dữ liệu ground truth với cột `EventTemplate`.
        df_parsedlog (pd.DataFrame): DataFrame chứa kết quả phân tích log với cột `EventTemplate`.
        filter_templates (list, optional): Danh sách các template cần lọc. Nếu không được cung cấp, tất cả các mẫu sẽ được sử dụng.

    Returns:
        tuple: (GA, FGA), trong đó:
            - GA (float): Độ chính xác nhóm (Grouping Accuracy).
            - FGA (float): F-Measure của độ chính xác nhóm.
    """ 
    null_logids = df_groundtruth[~df_groundtruth['EventTemplate'].isnull()].index       # Lấy các chỉ số (index) không null từ df_groundtruth 
    df_groundtruth = df_groundtruth.loc[null_logids]                                    # Lọc df_groundtruth để chỉ giữ lại các dòng không null 
    df_parsedlog = df_parsedlog.loc[null_logids]                                        # Tương tự
    GA, FGA = get_accuracy(df_groundtruth['EventTemplate'], df_parsedlog['EventTemplate'])
    print('Grouping_Accuracy (GA): %.4f, FGA: %.4f,'%(GA, FGA))
    return GA, FGA

def get_accuracy(series_groundtruth, series_parsedlog, filter_templates=None):
    """ Tính toán các chỉ số đánh giá độ chính xác giữa kết quả phân tích log và ground truth. Phương thức này tính toán hai chỉ số chính, gồm GA, FGA.

    Args:
        series_groundtruth (pandas.Series): Chuỗi chứa các template ground truth.
        series_parsedlog (pandas.Series): Chuỗi chứa các template từ kết quả phân tích log.
        filter_templates (list, optional): Danh sách các template cần lọc. Nếu không được cung cấp, tất cả các template sẽ được sử dụng để đánh giá độ chính xác. Nó được sử dụng để lọc các templates cụ thể trong quá trình tính toán độ chính xác. Nó cho phép người dùng chỉ tập trung vào một tập hợp con các template thay vì toàn bộ dữ liệu.

    Returns:
        tuple: (GA, FGA), trong đó:
            - GA (float): Độ chính xác nhóm (Grouping Accuracy).
            - FGA (float): F-Measure của độ chính xác nhóm.

    Example:
        >>> series_groundtruth = pd.Series(['A', 'B', 'A', 'C'])
        >>> series_parsedlog = pd.Series(['A', 'B', 'A', 'D'])
        >>> get_accuracy(series_groundtruth, series_parsedlog)
        (0.75, 0.6667)
    """
    
    series_groundtruth_valuecounts = series_groundtruth.value_counts()      # dtype: int64, A:3, B:2, C:1
    series_parsedlog_valuecounts = series_parsedlog.value_counts()          # Tương tự
    df_combined = pd.concat([series_groundtruth, series_parsedlog], axis=1, keys=['groundtruth', 'parsedlog'])
    grouped_df = df_combined.groupby('groundtruth')                         # Kết hợp và nhóm dữ liệu theo cột 'groundtruth'
    accurate_events = 0                                                     # Số lượng sự kiện chính xác          
    accurate_templates = 0                                                  # Số lượng template chính xác
    if filter_templates is not None:
        filter_identify_templates = set()                                   # Tập hợp lưu trữ các template đã được xác định
    
    for ground_truthId, group in tqdm(grouped_df):
        series_parsedlog_logId_valuecounts = group['parsedlog'].value_counts()          # Lấy số lượng các template phân tích được bởi công cụ cùng một nhóm với một template trong ground_truth 
        if filter_templates is not None and ground_truthId in filter_templates:         # Nếu template này đã có trong filter_templates và filter_templates không phải là None
            for parsed_eventId in series_parsedlog_logId_valuecounts.index:             # ==> parsed_eventId = "A" hoặc "B"
                filter_identify_templates.add(parsed_eventId)
        if series_parsedlog_logId_valuecounts.size == 1:
            parsed_eventId = series_parsedlog_logId_valuecounts.index[0]
            if len(group) == series_parsedlog[series_parsedlog == parsed_eventId].size: # Nếu có một template duy nhất trong nhóm và số lượng dòng trong nhóm bằng với số lượng dòng trong series_parsedlog ==> Nhóm log đúng
                if (filter_templates is None) or (ground_truthId in filter_templates):
                    accurate_events += len(group)
                    accurate_templates += 1
    if filter_templates is not None:
        GA = float(accurate_events) / len(series_groundtruth[series_groundtruth.isin(filter_templates)])        # Tính toán dựa trên số lượng trong filter_templates (phương thức isin() trả về True nếu giá trị trong series nằm trong filter_templates)
        PGA = float(accurate_templates) / len(filter_identify_templates)
        RGA = float(accurate_templates) / len(filter_templates)
    else:
        GA = float(accurate_events) / len(series_groundtruth)
        PGA = float(accurate_templates) / len(series_parsedlog_valuecounts)
        RGA = float(accurate_templates) / len(series_groundtruth_valuecounts)
    # print(FGA, RGA)
    FGA = 0.0
    if PGA != 0 or RGA != 0:
        FGA = 2 * (PGA * RGA) / (PGA + RGA)
    return GA, FGA

In [ ]:
# File PA_calculator.py trong UNLEASH
"""
This file is part of TA-Eval-Rep.
Copyright (C) 2022 University of Luxembourg
    This program is free software: you can redistribute it and/or modify
    it under the terms of the GNU General Public License as published by
    the Free Software Foundation, version 3 of the License.

    This program is distributed in the hope that it will be useful,
    but WITHOUT ANY WARRANTY; without even the implied warranty of
    MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
    GNU General Public License for more details.
    You should have received a copy of the GNU General Public License
    along with this program.  If not, see <https://www.gnu.org/licenses/>.
"""

import pandas as pd
import regex as re


def post_process_tokens(tokens, punc):
    """ Phương thức thực hiện hậu xử lý danh sách các token cho trước, loại bỏ các ký tự không cần thiết và chuẩn hóa các token.
    Chức năng chính:
        1. Nếu một token chứa chuỗi "<*>", toàn bộ token sẽ được thay thế bằng "<*>".
        2. Với các token khác, loại bỏ các ký tự không thuộc danh sách `punc`, không phải khoảng trắng (' '), 
           hoặc không nằm trong danh sách ký tự đặc biệt `excluded_str` (gồm '=', '|', '(', ')').
           
    Args:
        tokens (list): Danh sách các token cần xử lý.
        punc (str): Chuỗi chứa các ký tự phân cách và ký tự không cần thiết.    
    
    Returns:
        list: Danh sách các token đã được xử lý.
        
    Examples:
        >>> tokens = ["hello", "world<*>", "test|case", "a(b)c"]
        >>> punc = "!\"#$%&'()+,-/:;=?@[\\]^_`{|}~"
        >>> post_process_tokens(tokens, punc)
        ['hello', '<*>', 'test|case', 'abc']
    """
    excluded_str = ['=', '|', '(', ')']                         # Các ký tự đặc biệt cần giữ lại
    for i in range(len(tokens)):
        if tokens[i].find("<*>") != -1:
            tokens[i] = "<*>"
        else:
            new_str = ""
            for s in tokens[i]:
                if (s not in punc and s != ' ') or s in excluded_str:
                    new_str += s
            tokens[i] = new_str
    return tokens

def message_split(message):
    """ Tách chuỗi đầu vào thành các token dựa trên các ký tự phân cách và thực hiện xử lý hậu kỳ. 
    Chức năng chính:
        1. Tách chuỗi dựa trên các ký tự phân cách (khoảng trắng, dấu câu, ...).
        2. Loại bỏ các token rỗng hoặc không hợp lệ.
        3. Chuẩn hóa các token bằng cách loại bỏ các ký tự không cần thiết.
        4. Loại bỏ các token `<*>` liên tiếp.

    Args:
        message (str): Chuỗi đầu vào cần tách.

    Returns:
        list: Danh sách các token đã được xử lý.

    Examples:
        >>> message = "Hello, world! This is a test <*> <*>."
        >>> message_split(message)
        ['Hello', ',', 'world', '!', 'This', 'is', 'a', 'test', '<*>', '.']
    """
    
    punc = "!\"#$%&'()+,-/:;=?@.[\]^_`{|}~"                     # Các ký tự được sử dụng để phân tách
    splitters = "\s\\" + "\\".join(punc)                        # Tạo biểu thức chính quy để tách chuỗi
    splitter_regex = re.compile("([{}]+)".format(splitters))    
    
    # Ex: message = "Hello, world! This is a test <*> <*>."
    tokens = re.split(splitter_regex, message)          # tokens = ['Hello', ',', 'world', '!', 'This', 'is', 'a', 'test', '<*>', '<*>', '.']
    tokens = list(filter(lambda x: x != "", tokens))    # tokens = ['Hello', ',', 'world', '!', 'This', 'is', 'a', 'test', '<*>', '<*>', '.']
    
    tokens = post_process_tokens(tokens, punc)          # tokens = ['Hello', 'world', 'This', 'is', 'a', 'test', '<*>', '<*>']
    tokens = [ 
        token.strip() 
        for token in tokens 
        if token != "" and token != ' ' 
    ] # Loại bỏ các token rỗng hoặc khoảng trắng
    tokens = [ 
        token 
        for idx, token in enumerate(tokens) 
        if not (token == "<*>" and idx > 0 and tokens[idx - 1] == "<*>")
    ] # Loại bỏ các token `<*>` liên tiếp
    return tokens


def calculate_similarity(template1, template2):
    """ Phương thức đo lường mức độ giống nhau giữa hai chuỗi văn bản (template1 và template2) bằng cách sử dụng Chỉ số Jaccard.
    Chỉ số Jaccard là tỷ lệ giữa số lượng phần tử chung của hai tập hợp và tổng số phần tử của cả hai tập hợp.
    
    Args:
        template1 (str): Chuỗi văn bản đầu tiên.
        template2 (str): Chuỗi văn bản thứ hai. 
        
    Returns:
        float: Chỉ số Jaccard giữa hai chuỗi văn bản.    
    """
    template1 = message_split(template1)
    template2 = message_split(template2)
    intersection = len(set(template1).intersection(set(template2)))             # Giao giữa hai template
    union = (len(template1) + len(template2)) - intersection                    # Hợp giữa hai template
    return intersection / union


def calculate_parsing_accuracy(groundtruth_df, parsedresult_df, filter_templates=None):
    """ Tính toán độ chính xác của quá trình phân tích cú pháp (Parsing Accuracy - PA). 
    Quy trình hoạt động:
        1. Nếu `filter_templates` được cung cấp, lọc dữ liệu thực tế và kết quả phân tích để chỉ giữ lại các template trong danh sách này.
        2. So sánh cột `EventTemplate` giữa hai DataFrame để đếm số lượng message được phân tích đúng.
        3. Tính toán độ chính xác phân tích cú pháp (PA) bằng cách chia số lượng message đúng cho tổng số message.
        4. In ra độ chính xác phân tích cú pháp với định dạng 4 chữ số thập phân.
    
    Args:
        groundtruth_df (pd.DataFrame): DataFrame chứa dữ liệu thực tế với cột `EventTemplate` và `Content`.
        parsedresult_df (pd.DataFrame): DataFrame chứa kết quả phân tích với cột `EventTemplate` và `Content`.
        filter_templates (list, optional): Danh sách các template cần sử dụng để tính toán.

    Returns:
        float: Độ chính xác của quá trình phân tích cú pháp (Parsing Accuracy - PA), được tính bằng tỷ lệ giữa số lượng message được phân tích đúng và tổng số message.

    Examples:
        >>> groundtruth_df = pd.DataFrame({
        ...     'EventTemplate': ['A', 'B', 'C'],
        ...     'Content': ['msg1', 'msg2', 'msg3']
        ... })
        >>> parsedresult_df = pd.DataFrame({
        ...     'EventTemplate': ['A', 'B', 'D'],
        ...     'Content': ['msg1', 'msg2', 'msg3']
        ... })
        >>> calculate_parsing_accuracy(groundtruth_df, parsedresult_df)
        Parsing_Accuracy (PA): 0.6667
        0.6667
    """
    if filter_templates is not None:            # Nếu có filter_templates, tính toán theo các template chỉ định
        groundtruth_df = groundtruth_df[groundtruth_df['EventTemplate'].isin(filter_templates)]
        parsedresult_df = parsedresult_df.loc[groundtruth_df.index]
        
    correctly_parsed_messages = parsedresult_df[['EventTemplate']].eq(groundtruth_df[['EventTemplate']]).values.sum()
    total_messages = len(parsedresult_df[['Content']])

    PA = float(correctly_parsed_messages) / total_messages

    print('Parsing_Accuracy (PA): {:.4f}'.format(PA))
    return PA

# Phương thức mới, tính toán độ tương đồng phân tích cú pháp:
def calculate_similarity_accuracy(groundtruth_df, parsedresult_df, filter_templates=None):
    """ Tính toán độ chính xác tương đồng giữa các template phân tích cú pháp và dữ liệu thực tế. Dùng được cho tất cả các trình phân tích cú pháp.
    
    Args:
        groundtruth_df (pd.DataFrame): DataFrame chứa dữ liệu thực tế với cột `EventTemplate` và `Content`.
        parsedresult_df (pd.DataFrame): DataFrame chứa kết quả phân tích với cột `EventTemplate` và `Content`.
        filter_templates (list, optional): Danh sách các template cần sử dụng để tính toán.

    Returns:
        float: Độ chính xác tương đồng giữa các template phân tích cú pháp và dữ liệu thực tế.

    Examples:
        >>> groundtruth_df = pd.DataFrame({
        ...     'EventTemplate': ['A', 'B', 'C'],
        ...     'Content': ['msg1', 'msg2', 'msg3']
        ... })
        >>> parsedresult_df = pd.DataFrame({
        ...     'EventTemplate': ['A', 'B', 'D'],
        ...     'Content': ['msg1', 'msg2', 'msg3']
        ... })
        >>> calculate_similarity_accuracy(groundtruth_df, parsedresult_df)
        Similarity_Accuracy (SA): 0.6667
        0.6667
    """
    if filter_templates is not None:
        groundtruth_df = groundtruth_df[groundtruth_df['EventTemplate'].isin(filter_templates)]
        parsedresult_df = parsedresult_df.loc[groundtruth_df.index]

    similarities = []
    for index in range(len(groundtruth_df)):
        similarities.append(calculate_similarity(groundtruth_df['EventTemplate'][index], parsedresult_df['EventTemplate'][index]))
    SA = sum(similarities) / len(similarities)
    print('Similarity_Accuracy (SA): {:.4f}'.format(SA))
    return SA

def calculate_parsing_accuracy_lstm(groundtruth_df, parsedresult_df, filter_templates=None):
    """ Tương tự, Tính toán độ chính xác của quá trình phân tích cú pháp (Parsing Accuracy - PA) cho các trình phân tích dựa trên ngữ nghĩa.
        
        Args:
            groundtruth_df (pd.DataFrame): DataFrame chứa dữ liệu thực tế với cột `EventTemplate` và `Content`.
            parsedresult_df (pd.DataFrame): DataFrame chứa kết quả phân tích với cột `EventTemplate` và `Content`.
            filter_templates (list, optional): Danh sách các template cần sử dụng để tính toán.

        Returns:
            float: Độ chính xác của quá trình phân tích cú pháp (Parsing Accuracy - PA), được tính bằng tỷ lệ giữa số lượng message được phân tích đúng và tổng số message.
        """
    if filter_templates is not None:
        groundtruth_df = groundtruth_df[groundtruth_df['EventTemplate'].isin(filter_templates)]
        parsedresult_df = parsedresult_df.loc[groundtruth_df.index]

    # Tương tự, nhưng thêm một phương thức tính toán (correct_lstm) để kiểm tra độ chính xác dành riêng cho các trình phân tích dựa trên ngữ nghĩa
    groundtruth_templates = list(groundtruth_df['EventTemplate'])
    parsedresult_templates = list(parsedresult_df['EventTemplate'])
    correctly_parsed_messages = 0
    for i in range(len(groundtruth_templates)):
        if correct_lstm(groundtruth_templates[i], parsedresult_templates[i]):
            correctly_parsed_messages += 1

    PA = float(correctly_parsed_messages) / len(groundtruth_templates)
    print('Parsing_Accuracy (PA): {:.4f}'.format(PA))
    return PA

def correct_lstm(groudtruth, parsedresult):
    """ Phương thức tính toán độ chính xác phân tích dành riêng cho các trình phân tích cú pháp dựa trên ngữ nghĩa. Bản chất, chỉ chỉnh sửa lại, lọc các nhiễu trong groudtruth để so sánh với parsedresult.

    Args:
        groudtruth (str): Chuỗi văn bản gốc (ground truth).
        parsedresult (str): Chuỗi văn bản đã được phân tích (parsed result).

    Returns:
        bool: Trả về True nếu hai danh sách từ giống nhau, ngược lại trả về False.
    """
    tokens1 = groudtruth.split(' ')
    tokens2 = parsedresult.split(' ')
    tokens1 = [
        "<*>" 
        if "<*>" in token else token 
        for token in tokens1
    ]       # Chỉnh sửa lại token trong groudtruth
    return tokens1 == tokens2


In [ ]:
# File common.py trong UNLEASH
from __future__ import print_function

import time
import regex as re
import os
import pandas as pd
import numpy as np
import argparse
from datetime import datetime
from natsort import natsorted


all_datasets = [
    "Proxifier",
    "Linux",
    "Apache",
    "Zookeeper",
    "Mac",
    "OpenStack",
    "HealthApp",
    "Hadoop",
    "HPC",
    "OpenSSH",
    "BGL",
    "HDFS",
    # "Android",
    "Spark",
    # "Windows",
    "Thunderbird",
]

datasets = ['HDFS', 'Hadoop', 'Spark', 'Zookeeper', 'OpenStack', 'BGL', 'HPC', 'Thunderbird', 'Windows', 'Linux', 'Mac', 'Android', 'HealthApp', 'Apache', 'OpenSSH', 'Proxifier']  


def sort_templates(templates):
    """ Sắp xếp danh sách các template theo độ dài giảm dần.
    Args:
        templates (list): Danh sách các template cần sắp xếp.
    Returns:
        list: Danh sách các template đã được sắp xếp theo thứ tự giảm dần của độ dài.
    """
    return sorted(templates, key=lambda x: len(x), reverse=True)


def get_pattern_from_template(template):
    """ Tạo một biểu thức chính quy từ một chuỗi template. Chuỗi template có thể chứa các ký tự đặc biệt và ký tự đại diện `<*>`. Hàm này thực hiện:
    - Escape các ký tự đặc biệt trong template để đảm bảo chúng không bị hiểu nhầm.
    - Thay thế các khoảng trắng bằng biểu thức chính quy `\s+` để khớp với một hoặc nhiều khoảng trắng.
    - Thay thế ký tự đại diện `<*>` bằng `(\S+?)` để khớp với một hoặc nhiều ký tự không phải khoảng trắng.
        - Thêm dấu `^` và `$` để biểu thức chính quy khớp toàn bộ chuỗi.

    Args:
        template (str): Chuỗi mẫu cần chuyển đổi thành biểu thức chính quy.

    Returns:
        str: Biểu thức chính quy được tạo từ chuỗi mẫu.
    Examples:
        >>> get_pattern_from_template("File <*> not found")
        '^File\\s+(\\S+?)\\s+not\\s+found$'
        >>> get_pattern_from_template("User <*> logged in at <*>")
        '^User\\s+(\\S+?)\\s+logged\\s+in\\s+at\\s+(\\S+?)$'
    """
    escaped = re.escape(template)                       # Thoát (escape) các ký tự đặc biệt trong template
    spaced_escape = re.sub(r'\\\s+', "\\\s+", escaped)  
    return "^" + spaced_escape.replace(r"<\*>", r"(\S+?)") + "$"  # Thêm dấu `^` và `$` để biểu thức chính quy khớp toàn bộ chuỗi, một <*> duy nhất có thể sử dụng nhiều message.

def is_abstract(x, y):
    """ Xác định m template `x` có trừu tượng hơn template `y` hay không. Hàm này kiểra xem template `y` có khớp với biểu thức chính quy được tạo từ template `x` hay không.

    Args:
        x (str): Chuỗi mẫu (template).
        y (str): Chuỗi cần kiểm tra (template hoặc message).

    Returns:
        bool: Trả về `True` nếu `x` trừu tượng hơn `y`, ngược lại trả về `False`.

    Examples:
        >>> x = "Hello <*>"
        >>> y = "Hello world"
        >>> is_abstract(x, y)  # Trả về True

        >>> y = "Hi world"
        >>> is_abstract(x, y)  # Trả về False
    """

    if y is np.nan:
        return False

    m = re.match(get_pattern_from_template(x), y)
    if m:
        return True
    else:
        return False


def common_args():
    parser = argparse.ArgumentParser()
    parser.add_argument('-otc', '--oracle_template_correction',
                        help="Set this if you want to use corrected oracle templates",
                        default=False, action='store_true')
    parser.add_argument('-full', '--full_data',
                        help="Set this if you want to test on full dataset",
                        default=False, action='store_true')
    parser.add_argument('--complex', type=int,
                        help="Set this if you want to test on complex dataset",
                        default=0)
    parser.add_argument('--frequent', type=int,
                        help="Set this if you want to test on frequent dataset",
                        default=0)
    parser.add_argument('--shot', type=int,
                        help="Set this if you want to test on complex dataset",
                        default=0)
    parser.add_argument('--example_size', type=int,
                        help="Set this if you want to test on frequent dataset",
                        default=0)    
    args = parser.parse_args()
    return args


def unique_output_dir(name):
    """ Tạo một đường dẫn tục đầu ra duy nhất dựa trên tên cơ sở, thời gian hiện tại và ID tiến trình.

    Args:
        name (str): Tên cơ sở của thư mục.

    Returns:
        str: Đường dẫn thư mục đầu ra duy nhất.

    Examples:
        >>> uniqcue_output_dir("experiment")
        'experiment_result/20231005_153045_12345'
    """
    return os.path.join('{}_result'.format(name), '{}_{}'.format(datetime.now().strftime("%Y%m%d_%H%M%S"), os.getpid()))


def correct_single_template(template, user_strings=None):
    """ Áp dụng các quy tắc để xử lý và chuẩn hóa một chuỗi template.Dựa trên bài báo `Guidelines for assessing the accuracy of log message template identification techniques`. 
    DOI: 10.10003.3510101.
    
    Các quy tắc được áp dụng:
        - **DS (Double Space):** Thay thế nhiều khoả*ng trắng bằng một khoảng trắng duy nhất. `Input:  <*>` --> `Input:  <*>`   
        - **BL (Boolean):** Thay thế các giá trị boolean (true, false) bằng `<*>`. `cancel=false` --> `cancel=<*>`
        - **US (User String):** Thay thế các chuỗi người dùng (user strings) bằng `<*>`. `username=admin` --> `username=<*>`
        - **DG (Digit):** Thay thế các chữ số bằng `<*>`. `count=0` --> `count=<*>`
        - **PS (Path-like tring):** Thay thế các chuỗi giống đường dẫn bằng `<*>`. `/usr/local/bin started` --> `<*> started`
        - **WV (Word concatenated with Variable, hay Mixed Token (MT)):** Thay thế các từ kết hợp với biến bằng `<*>`. `python v<*>` --> python <*>`
        - **DV (Dot-separated Variables):** Thay thế các biến được phân tách bằng dấu chấm bằng `<*>`. `<*>.<*> seconds` --> `<*> seconds`
        - **CV (Consecutive Variables):** Thay thế các biến liên tiếp bằng `<*>`. `value=<*><*>` --> `value=<*>`
    Args:
        template (str): Chuỗi template cần xử lý.
        user_strings (set, optional): Tập hợp các chuỗi do người dùng định nghĩa cần thay thế bằng `<*>`.

    Returns:
        str: Chuỗi template đã được xử lý và chuẩn hóa.

    Examples:
        >>> template = "Received block <*><*> of size blk_<*> from v<*>"
        >>> correct_single_template(template)
        'Received block <*> of size <*> from <*>'
    """

    boolean = {'true', 'false'}                 # Tập hợp các chuỗi đại diện boolean (BL)          
    default_strings = {'null', 'root', 'admin'} # Các chuỗi do người dùng định nghĩa (US), mặc định là 3 gtrị này (US)
    path_delimiters = {                         # Các ký tự sử dụng phân tách để nhận diện đường dẫn (PS)
        r'\s', r'\,', r'\!', r'\;', r'\:',
        r'\=', r'\|', r'\"', r'\'',
        r'\[', r'\]', r'\(', r'\)', r'\{', r'\}'
    }
    token_delimiters = path_delimiters.union({  # Tập hợp tất cả các dấu phân cách để phân chia các quy tắc còn lại (DG, WV, DV, CV, MT)
        r'\.', r'\-', r'\+', r'\@', r'\#', r'\$', r'\%', r'\&',
    })

    if user_strings:
        default_strings = default_strings.union(user_strings) # Thêm các chuỗi do người dùng định nghĩa vào tập hợp các chuỗi mặc định

    # Áp dụng DS (Double Space)
    template = template.strip()
    template = re.sub(r'\s+', ' ', template)

    # Áp dụng PS (Path String)
    p_tokens = re.split('('+'|'.join(path_delimiters)+')', template)
    new_p_tokens = []
    for p_token in p_tokens:
        if re.match(r'^(\/[^\/]+)+$', p_token):
            p_token = '<*>'
        new_p_tokens.append(p_token)
    template = ''.join(new_p_tokens)        # "/path/to/file, another/path" --> "<*>, another/path"

    # Áp dụng các quy tắc còn lại:
    tokens = re.split('('+'|'.join(token_delimiters)+')', template)  # Phân chia thành các token trong khi vẫn giữ các dấu phân cách
    new_tokens = []
    for token in tokens:
        # Áp dụng BL, US (Boolean, User String)
        for to_replace in boolean.union(default_strings):
            if token.lower() == to_replace.lower():
                token = '<*>'

        # Áp dụng DG (Digit)
        if re.match(r'^\d+$', token):
            token = '<*>'

        # Áp dụng WV (Word concatenated with Variable), hay MT (Mixed Token)
        if re.match(r'^[^\s\/]*<\*>[^\s\/]*$', token):
            if token != '<*>/<*>':  # need to check this because `/` is not a deliminator
                token = '<*>'

        # Lấy kết quả
        new_tokens.append(token)

    # Tạo lại template chuẩn từ các token đã xử lý
    template = ''.join(new_tokens)

    # Chỉ thay thế các biến liên tiếp nếu được phân tách bằng bất kỳ phân tách nào bao gồm "." (DV)
    while True:
        prev = template
        template = re.sub(r'<\*>\.<\*>', '<*>', template)
        if prev == template:
            break

    # Chỉ thay thế các biến liên tiếp nếu không được phân tách bằng bất kỳ dấu phân cách nào bao gồm cả khoảng trắng (CV)
    # NOTE: Nên được thực hiện ở cuối cùng vì nó có thể thay thế các biến đã được thay thế trước đó
    while True:
        prev = template
        template = re.sub(r'<\*><\*>', '<*>', template)
        if prev == template:
            break

    return template


def correct_templates_and_update_files(dir_path, log_file_basename, inplace=False):
    """ Hàm xử lý cập nhật log có cấu trúc và file mẫu (template) sau khi áp dụng các quy tắc sửa mẫu.
    - Nếu một mẫu được sửa, chỉ cập nhật cột 'EventTemplate' trong structured log.
    - Nếu nhiều mẫu được gộp thành một, cần cập nhật cả 'EventId' và 'EventTemplate' trong log.
    - Trong trường hợp gộp, dùng EventId đầu tiên trong nhóm làm đại diện để giữ đồng nhất.

    Args:
        dir_path (str): Đường dẫn đến thư mục chứa các file log.
        log_file_basename (str): Tên cơ sở của file log.
        inplace (bool): Nếu True, cập nhật file log gốc. Nếu False, tạo file mới với tên '_corrected'.
    
    Returns:
        None: Hàm không trả về giá trị nào, nhưng sẽ tạo các file mới hoặc cập nhật file gốc.    
    
    Examples:
        >>> <Mẫu gốc (EventId, EventTemplate)>
            E1, Send 123
            E2, Send 456
        .
        >>> <Structured log gốc>
            1, Send 123, E1, Send 123
            2, Send 456, E2, Send 456
        .
        >>> <Mẫu sau khi gộp>
            E1, Send <*>
        .
        >>> <Structured log mong đợi>
            1, Send 123, E1, Send <*>
            2, Send 456, E1, Send <*>
    """

    # Đường dẫn đến file structured log gốc
    org_structured_log_file = os.path.join(dir_path, log_file_basename + '_structured.csv')

    # Chuyển đổi tên file log gốc thành DataFrame
    structured_logs_df = pd.read_csv(org_structured_log_file)

    # Trích xuất templates_df
    # Loại bỏ các dòng trùng lặp dựa trên cột EventTemplate. Sau đó, loại bỏ các dòng có giá trị NaN trong cột EventTemplate bằng dropna, (điều này cần thiết vì một số công cụ như LogSig có thể tạo ra các mẫu rỗng).
    templates_df = structured_logs_df.drop_duplicates(subset='EventTemplate').dropna(subset=['EventTemplate'])

    # Chuyển đổi DataFrame templates_df thành dictionary (key: EventId, value: EventTemplate)
    templates_dict = templates_df.set_index('EventId')['EventTemplate'].to_dict()

    #  Áp dụng 8 quy tắc sửa mẫu và gộp template
    new_templates_dict = correct_templates(templates_dict)

    # Cập nhật DataFrame structured_logs_df
    start_time = time.time()
    for index, row in structured_logs_df.iterrows():

        # Duyệt qua từng dòng trong structured_logs_df. Tìm template tương ứng với EventId trong new_templates_dict.
        is_matched = False
        for tids, template in new_templates_dict.items():
            if row['EventId'] in tids: # Nếu tìm thấy, cập nhật EventId và EventTemplate trong DataFrame.
                structured_logs_df.at[index, 'EventId'] = tids[0]
                structured_logs_df.at[index, 'EventTemplate'] = template
                is_matched = True
                break

        # Nếu không tìm thấy, in cảnh báo với thông tin EventId và nội dung (Content) của dòng đó.
        if is_matched is False:
            print('*** WARN: No matching template; EventId:', row['EventId'], 'message:', row['Content'])

    # Cập nhật file structured log
    if inplace:  # Nếu inplace=True, Ghi đè file gốc
        structured_logs_df.to_csv(org_structured_log_file, index=False)
    else:        # Nếu inplace=False, tạo file mới với hậu tố '_corrected'
        structured_logs_df.to_csv(os.path.join(dir_path, log_file_basename + '_structured_corrected.csv'), index=False)

    # Cập nhật file templates
    new_templates = []
    for tids, template in new_templates_dict.items():
        new_templates.append((tids[0], template))
    new_templates = natsorted(new_templates, key=lambda x: x[0])
    new_templates_df = pd.DataFrame(new_templates, columns=['EventId', 'EventTemplate'])
    new_templates_df.to_csv(os.path.join(dir_path, log_file_basename + '_templates_corrected.csv'), index=False)

    print('Structured log and templates file update done. [Time taken: {:.3f}]'.format(time.time() - start_time))

def correct_templates(templates_dict):
    """ Xử lý hậu kỳ (post-processing) các templates để chuẩn hóa hoặc hợp nhất các template tương tự. 

    Args:
        templates_dict (dict): Từ điển chứa các template gốc với định dạng {EventId: EventTemplate}.
    
    Returns:
        dict: Từ điển chứa các template đã được xử lý với định dạng {(EventId1, EventId2, ...): EventTemplate}.
    
    Examples:
        >>> Input:
            templates_dict = {
                "E1": "Send 123",
                "E2": "Send 456",
                "E3": "Receive 789"
            }

        >>> Output:
            new_templates_dict = {
                ("E1", "E2"): "Send <*>",
                ("E3",): "Receive <*>"
            }
    """

    # Các template bị ảnh hưởng sau khi xử lý
    change_count = 0             # Số lượng template bị thay đổi
    inverse_templates_dict = {}  # key: EventTemplate, value: list of EventIds

    start_time = time.time()
    for tid, template in sorted(templates_dict.items(), key=lambda x: x[0]):  # sort to avoid non-determinism
        org_template = template
        new_template = correct_single_template(template)

        # Đếm số lượng mẫu bị thay đổi
        if org_template != new_template:
            change_count += 1

        # Cập nhật inverse_templates_dict
        if new_template in inverse_templates_dict.keys():
            inverse_templates_dict[new_template].append(tid) # Nếu template đã tồn tại, thêm EventId vào danh sách
        else:
            inverse_templates_dict[new_template] = [tid]     # Nếu chưa tồn tại, khởi tạo danh sách mới

    # Xây dựng new_templates_dict từ inverse_templates_dict
    new_templates_dict = {tuple(tids): template for template, tids in inverse_templates_dict.items()}

    end_time = time.time() - start_time
    print('\tOriginal templates:', len(templates_dict.keys()))
    print('\tTemplates after correction:', len(new_templates_dict.keys()))
    print("\tTemplates changed by correction:", change_count)
    print('Template correction done. [Time taken: {:.3f}]'.format(end_time))

    return new_templates_dict


In [ ]:
# File evaluator.py trong LogGzip
import pandas as pd
import numpy as np
from tqdm import tqdm
from scipy.special import comb
from sklearn.metrics import accuracy_score
import regex as re
import sys


def post_process_tokens(tokens, punc):
    """
    Phương thức xử lý danh sách token để loại bỏ các ký tự không cần thiết, sau đó chuẩn hóa lại các token.
    Cụ thể, phương thức duyệt qua từng token, nếu chứa <*>, thay thế toàn bộ bằng <*>. Sau đó, loại bỏ dấu câu được xác định trong punc trừ một số ký tự đặc biệt ['=', '|', '(', ')']. Cuối cùng, trả về danh sách token đã xử lý.
    Args:
        tokens (list): Danh sách các token đã được tách ra từ chuỗi đầu vào.
        punc (str): Chuỗi chứa các ký tự được sử dụng để loại bỏ.
    
    Returns:
        list: Danh sách các token đã được xử lý và chuẩn hóa.
    """
    excluded_str = ['=', '|', '(', ')']         # Các ký tự đặc biệt không loại bỏ.
    for i in range(len(tokens)):
        if tokens[i].find("<*>") != -1:
            tokens[i] = "<*>"                   # Ex: blk_<*> --> <*>
        else:
            # Loại bỏ các ký tự không cần thiết trong token theo danh sách punc.
            # Default: punc = "!\"#$%&'()+,-/:;=?@.[\]^_`{|}~"
            new_str = ""
            for s in tokens[i]:
                if (s not in punc and s != ' ') or s in excluded_str:
                    new_str += s
            tokens[i] = new_str
    return tokens

def message_split(message):
    """
    Chia một chuỗi đầu vào thành danh sách các token dựa trên khoảng trắng và các dấu câu đặc biệt. 
    Cụ thể, (1) Xác định các ký tự phân tách (dấu câu, khoảng trắng); (2) Sử dụng biểu thức chính quy để tách chuỗi; (3) Loại bỏ các token rỗng hoặc chỉ chứa khoảng trắng; (4)Tiền xử lý token để loại bỏ ký tự không mong muốn; (5) Xử lý trường hợp có nhiều `<*>` liên tiếp.
    Args:
        message (str): Chuỗi đầu vào cần tách.

    Returns:
        list: Danh sách các token sau khi tách.
    
    Example:
        >>> message_split("Hello, world! How are you?")
        ['Hello', ',', 'world', '!', 'How', 'are', 'you', '?']
    """
    punc = "!\"#$%&'()+,-/:;=?@.[\]^_`{|}~"                     # Các ký tự được sử dụng để tách chuỗi.
    splitters = "\s\\" + "\\".join(punc)
    splitter_regex = re.compile("([{}]+)".format(splitters))    # Tạo regex để tách chuỗi, "([{}]+)" sẽ tìm các ký tự trong splitters và giữ chúng lại trong kết quả tách.
    tokens = re.split(splitter_regex, message)                  # Tách chuỗi: "Hello,,, world.. How are you?" --> ["Hello", ",,,", "world", "..", "How", "are", "you", "?"]
    tokens = list(filter(lambda x: x != "", tokens))            # Loại bỏ các token rỗng.
    tokens = post_process_tokens(tokens, punc)                  # Xử lý hậu kỳ
    tokens = [token.strip() for token in tokens if token != "" and token != ' ']        # Loại bỏ các token rỗng và khoảng trắng.
    tokens = [token for idx, token in enumerate(tokens) if not (token == "<*>" and idx > 0 and tokens[idx - 1] == "<*>")] # Loại bỏ các token "<*>" liên tiếp.
    return tokens

def calculate_similarity(template1, template2):
    """
    Phương thức đo lường mức độ giống nhau giữa hai chuỗi văn bản (template1 và template2) bằng cách sử dụng Chỉ số Jaccard.
    Chỉ số Jaccard là tỷ lệ giữa số lượng phần tử chung của hai tập hợp và tổng số phần tử của cả hai tập hợp.
    
    Args:
        template1 (str): Chuỗi văn bản đầu tiên.
        template2 (str): Chuỗi văn bản thứ hai. 
        
    Returns:
        float: Chỉ số Jaccard giữa hai chuỗi văn bản.    
    """
    template1 = message_split(template1)
    template2 = message_split(template2)
    intersection = len(set(template1).intersection(set(template2))) # Tính số lượng phần tử chung giữa hai tập hợp.
    union = (len(template1) + len(template2)) - intersection        # Tính tổng số phần tử của cả hai tập hợp.
    return intersection / union

def evaluate_template_level(dataset, df_groundtruth, df_parsedresult, filter_templates=None):
    """
    Phương thức đánh giá chất lượng của một hệ thống trích xuất template (EventTemplate) bằng cách so sánh kết quả phân tích (df_parsedresult) với dữ liệu gốc (df_groundtruth).
    
    Args:
        dataset (str): Tên của tập dữ liệu.
        df_groundtruth (pd.DataFrame): DataFrame chứa các template gốc.
        df_parsedresult (pd.DataFrame): DataFrame chứa các template đã được phân tích.
        filter_templates (list, optional): Danh sách các template cần lọc. Mặc định là None.
    
    Returns:
        tuple: (t1, t2, FTA, PTA, RTA), trong đó:
            - t1 (int): Số lượng mẫu được nhận diện.
            - t2 (int): Số lượng mẫu thực tế.
            - FTA (float): F1-score của mẫu trích xuất.
            - PTA (float): Precision (độ chính xác) của mẫu trích xuất.
            - RTA (float): Recall (độ phủ) của mẫu trích xuất.
    """
    correct_parsing_templates = 0
    if filter_templates is not None:
        filter_identify_templates = set()        # Lưu trữ tập hợp các mẫu được lọc (nếu có).
    null_logids = df_groundtruth[~df_groundtruth['EventTemplate'].isnull()].index
    
    # Loại bỏ các dòng có giá trị NaN trong cột EventTemplate của df_groundtruth.
    df_groundtruth = df_groundtruth.loc[null_logids]
    df_parsedresult = df_parsedresult.loc[null_logids]
    
    # Tạo các Series từ cột EventTemplate của df_groundtruth và df_parsedresult, đếm số lần xuất hiện của từng mẫu thực tế.
    series_groundtruth = df_groundtruth['EventTemplate']
    series_parsedlog = df_parsedresult['EventTemplate']
    series_groundtruth_valuecounts = series_groundtruth.value_counts()

    # Gộp dữ liệu từ df_groundtruth và df_parsedresult, nhóm theo parsedlog.
    df_combined = pd.concat([series_groundtruth, series_parsedlog], axis=1, keys=['groundtruth', 'parsedlog'])
    grouped_df = df_combined.groupby('parsedlog')

    for identified_template, group in tqdm(grouped_df):         # tqdm() thực hiện hiển thị tiến trình của vòng lặp.
        corr_oracle_templates = set(list(group['groundtruth']))
        if filter_templates is not None and len(corr_oracle_templates.intersection(set(filter_templates))) > 0:
            filter_identify_templates.add(identified_template)

        if corr_oracle_templates == {identified_template}:
            if (filter_templates is None) or (identified_template in filter_templates):
                correct_parsing_templates += 1

    if filter_templates is not None:
        PTA = correct_parsing_templates / len(filter_identify_templates)
        RTA = correct_parsing_templates / len(filter_templates)
    else:
        PTA = correct_parsing_templates / len(grouped_df)
        RTA = correct_parsing_templates / len(series_groundtruth_valuecounts)
    FTA = 0.0
    if PTA != 0 or RTA != 0:
        FTA = 2 * (PTA * RTA) / (PTA + RTA)
    print('PTA: {:.4f}, RTA: {:.4f} FTA: {:.4f}'.format(PTA, RTA, FTA))
    t1 = len(grouped_df) if filter_templates is None else len(filter_identify_templates)
    t2 = len(series_groundtruth_valuecounts) if filter_templates is None else len(filter_templates)
    print("Identify : {}, Groundtruth : {}".format(t1, t2))
    return t1, t2, FTA, PTA, RTA

def correct_lstm(groundtruth, parsedresult):
    """
    Phương thức này kiểm tra xem hai chuỗi groundtruth (chuẩn) và parsedresult (kết quả đã phân tích) có giống nhau hay không, nhưng có một điều kiện đặc biệt: Nếu một token trong groundtruth chứa "<*>", thì toàn bộ token đó sẽ được thay thế bằng "<*>", sau đó mới so sánh hai danh sách token.
    
    Args:
        groundtruth (str): Chuỗi đầu vào gốc.
        parsedresult (str): Chuỗi đầu vào đã được phân tích.    
    
    Returns:
        bool: True nếu hai chuỗi giống nhau sau khi xử lý, False nếu không.
    """
    tokens1 = groundtruth.split(' ')
    tokens2 = parsedresult.split(' ')
    tokens1 = ["<*>" if "<*>" in token else token for token in tokens1]
    return tokens1 == tokens2

def calculate_parsing_accuracy(groundtruth_df, parsedresult_df, filter_templates=None):
    """
    Phương thức tính độ chính xác của quá trình phân tích cú pháp (Parsing Accuracy - PA) dựa trên số lượng message được phân tích đúng so với tổng số message.
    
    Args:
        groundtruth_df (pd.DataFrame): DataFrame chứa các template gốc.
        parsedresult_df (pd.DataFrame): DataFrame chứa các template đã được phân tích.
        filter_templates (list, optional): Danh sách các template cần lọc. Mặc định là None.
        
    Returns:
        float: Độ chính xác của quá trình phân tích cú pháp (PA).
    """
    if filter_templates is not None:
        groundtruth_df = groundtruth_df[groundtruth_df['EventTemplate'].isin(filter_templates)]
        parsedresult_df = parsedresult_df.loc[groundtruth_df.index]
    correctly_parsed_messages = parsedresult_df[['EventTemplate']].eq(groundtruth_df[['EventTemplate']]).values.sum()
    total_messages = len(parsedresult_df[['Content']])
    PA = float(correctly_parsed_messages) / total_messages
    print('Parsing_Accuracy (PA): {:.4f}'.format(PA))
    return PA

def calculate_parsing_accuracy_lstm(groundtruth_df, parsedresult_df, filter_templates=None):
    """
    Phương thức `calculate_parsing_accuracy_lstm` được sử dụng để tính độ chính xác của việc phân tích cú pháp (Parsing Accuracy - PA) giữa dữ liệu thực tế (groundtruth_df) và dữ liệu kết quả được phân tích (parsedresult_df). Phương thức này đặc biệt sử dụng trong bối cảnh mô hình LSTM để phân tích template (Event Templates).
    
    Args:
        groundtruth_df (pd.DataFrame): DataFrame chứa các template gốc.
        parsedresult_df (pd.DataFrame): DataFrame chứa các template đã được phân tích.
        filter_templates (list, optional): Danh sách các template cần lọc. Mặc định là None.
        
    Returns:
        float: Độ chính xác của quá trình phân tích cú pháp (PA).
    """
    # parsedresult_df = pd.read_csv(parsedresult)
    # groundtruth_df = pd.read_csv(groundtruth)
    if filter_templates is not None:
        groundtruth_df = groundtruth_df[groundtruth_df['EventTemplate'].isin(filter_templates)]
        parsedresult_df = parsedresult_df.loc[groundtruth_df.index]
    # correctly_parsed_messages = parsedresult_df[['EventTemplate']].eq(groundtruth_df[['EventTemplate']]).values.sum()
    groundtruth_templates = list(groundtruth_df['EventTemplate'])
    parsedresult_templates = list(parsedresult_df['EventTemplate'])
    correctly_parsed_messages = 0
    for i in range(len(groundtruth_templates)):
        if correct_lstm(groundtruth_templates[i], parsedresult_templates[i]):
            correctly_parsed_messages += 1

    PA = float(correctly_parsed_messages) / len(groundtruth_templates)

    # similarities = []
    # for index in range(len(groundtruth_df)):
    #     similarities.append(calculate_similarity(groundtruth_df['EventTemplate'][index], parsedresult_df['EventTemplate'][index]))
    # SA = sum(similarities) / len(similarities)
    # print('Parsing_Accuracy (PA): {:.4f}, Similarity_Accuracy (SA): {:.4f}'.format(PA, SA))
    print('Parsing_Accuracy (PA): {:.4f}'.format(PA))
    return PA

def evaluate(groundtruth, parsedresult):
    df_groundtruth = pd.read_csv(groundtruth)
    df_parsedlog = pd.read_csv(parsedresult)
    
    # Remove invalid groundtruth event Ids
    non_empty_log_ids = df_groundtruth[~df_groundtruth["EventTemplate"].isnull()].index
    df_groundtruth = df_groundtruth.loc[non_empty_log_ids]
    df_parsedlog = df_parsedlog.loc[non_empty_log_ids]

    GA, FGA = get_accuracy(df_groundtruth["EventTemplate"], df_parsedlog["EventTemplate"])

    accuracy_exact_string_matching = accuracy_score(
        np.array(df_groundtruth.EventTemplate.values, dtype='str'),
        np.array(df_parsedlog.EventTemplate.values, dtype='str')
    )
    # PA = calculate_parsing_accuracy_lstm(df_groundtruth, df_parsedlog)


    _, _, FTA, PTA, RTA = evaluate_template_level(None, df_groundtruth, df_parsedlog)

    print(
        "Grouping_Accuracy (GA): {:.4f},  FGA: {:.4f}, FTA: {:.4f}, PTA: {:.4f}, RTA: {:.4f}".format(
            GA, FGA, FTA, PTA, RTA
        )
    )
    return GA, FGA, FTA, PTA, RTA

def get_accuracy(series_groundtruth, series_parsedlog, filter_templates=None):
    series_groundtruth_valuecounts = series_groundtruth.value_counts()
    series_parsedlog_valuecounts = series_parsedlog.value_counts()
    df_combined = pd.concat([series_groundtruth, series_parsedlog], axis=1, keys=['groundtruth', 'parsedlog'])
    grouped_df = df_combined.groupby('groundtruth')
    accurate_events = 0 # determine how many lines are correctly parsed
    accurate_templates = 0
    if filter_templates is not None:
        filter_identify_templates = set()
    for ground_truthId, group in tqdm(grouped_df):
        series_parsedlog_logId_valuecounts = group['parsedlog'].value_counts()
        if filter_templates is not None and ground_truthId in filter_templates:
            for parsed_eventId in series_parsedlog_logId_valuecounts.index:
                filter_identify_templates.add(parsed_eventId)
        if series_parsedlog_logId_valuecounts.size == 1:
            parsed_eventId = series_parsedlog_logId_valuecounts.index[0]
            if len(group) == series_parsedlog[series_parsedlog == parsed_eventId].size:
                if (filter_templates is None) or (ground_truthId in filter_templates):
                    accurate_events += len(group)
                    accurate_templates += 1
    if filter_templates is not None:
        GA = float(accurate_events) / len(series_groundtruth[series_groundtruth.isin(filter_templates)])
        PGA = float(accurate_templates) / len(filter_identify_templates)
        RGA = float(accurate_templates) / len(filter_templates)
    else:
        GA = float(accurate_events) / len(series_groundtruth)
        PGA = float(accurate_templates) / len(series_parsedlog_valuecounts)
        RGA = float(accurate_templates) / len(series_groundtruth_valuecounts)
    FGA = 0.0
    if PGA != 0 or RGA != 0:
        FGA = 2 * (PGA * RGA) / (PGA + RGA)
    return GA, FGA